In [ ]:
#@title 1. Setup - run once (about a minute)
import os, shutil, subprocess

REPO = "https://github.com/freelancermeer/ytmeer.git"
DIR  = "/content/ytmeer"

# ffmpeg merges the video+audio streams and reports the real quality, aria2c
# pulls each file over 16 connections, and node is the JS runtime yt-dlp needs
# for YouTube's "n" challenge. Colab already ships node.
!apt-get -qq install -y ffmpeg aria2 > /dev/null 2>&1
!pip -q install "yt-dlp>=2026.7.4" "gradio>=6,<7" "curl_cffi>=0.10,<0.16"

# The downloader itself - a pull if it is already here, a clone if not.
if os.path.isdir(os.path.join(DIR, ".git")):
    subprocess.run(["git", "-C", DIR, "pull", "-q"])
else:
    subprocess.run(["git", "clone", "-q", REPO, DIR])
os.makedirs("/content/downloads", exist_ok=True)

# OPTIONAL - write into Google Drive instead, so a disconnect does not take the
# files with it. Uncomment, re-run this cell, then set the output folder in the UI.
# from google.colab import drive; drive.mount("/content/drive")

for tool in ("yt-dlp", "ffmpeg", "ffprobe", "aria2c", "node"):
    print(f"  {tool:8} {shutil.which(tool) or 'MISSING'}")

print("""
Now run cell 2. It prints one URL - that is the UI and the API both, with a
copy-paste snippet for driving it from your own code.

  COOKIES ARE OPTIONAL. Public videos come down without an account. Upload a
  cookies.txt in the UI only if you want private / members-only / age-restricted
  videos, or if YouTube starts asking the runtime to confirm it is not a bot -
  the run says so plainly in its summary if that ever happens.

  WHERE THE FILES GO: /content/downloads, which Colab WIPES when the runtime
  disconnects. For a long run, mount Drive with the commented line above and set
  the output folder to /content/drive/MyDrive/yt-downloads in the UI.

  ERRORS are reported, not swallowed. Anything that does not come down is listed
  at the end of the log and in the API's "failures", with the reason.
""")

In [ ]:
#@title 2. Run - starts the UI and the API
import importlib, sys

sys.path.insert(0, "/content/ytmeer")
import colab_app
importlib.reload(colab_app)     # so a re-run picks up a git pull without restarting

# share=True is what makes the URL reachable from outside Colab, which is what
# the API needs. Leave this cell running while you use it.
colab_app.launch(share=True)